# Chapter 12: Digital Wallet
*System Design Interview Volume 2*

## TL;DR

A digital wallet for cross-wallet balance transfers at **1M TPS** requires distributed event sourcing. The design evolves through three stages: in-memory Redis (fast but not durable), relational DB with distributed transactions (correct but not auditable), and **event sourcing with CQRS** (correct, auditable, reproducible). File-based storage with mmap, RocksDB snapshots, and Raft consensus achieves high performance and reliability. Distributed transactions (TC/C or Saga) coordinate cross-partition transfers.

## Requirements

| Requirement | Detail |
|---|---|
| Balance transfer | Move money between two digital wallets on the same platform |
| Throughput | 1,000,000 TPS |
| Availability | 99.99% uptime |
| Transactional | Atomic transfers -- no partial updates |
| Reproducibility | Replay history to reconstruct any past balance state |

## Estimation

In [ ]:
# Back-of-the-envelope: Digital Wallet
target_tps = 1_000_000
ops_per_transfer = 2  # debit + credit

total_ops = target_tps * ops_per_transfer
print(f"Total operations/sec: {total_ops:,}")

# Node estimation at different per-node TPS
for node_tps in [100, 1_000, 10_000]:
    nodes = total_ops / node_tps
    print(f"  At {node_tps:>6,} TPS/node: {nodes:>7,.0f} nodes needed")

print()

# Event storage (1 day)
avg_event_bytes = 200  # account_id, amount, timestamp, type
events_per_day = total_ops * 86_400
storage_tb = events_per_day * avg_event_bytes / (1024**4)
print(f"Events/day: {events_per_day:,.0f}")
print(f"Event storage/day: {storage_tb:.1f} TB")

# Snapshot size (1B accounts, 100 bytes each)
accounts = 1_000_000_000
snapshot_gb = accounts * 100 / (1024**3)
print(f"Snapshot size: {snapshot_gb:.1f} GB")

## High-Level Design: Evolution

```mermaid
graph TB
    subgraph v1["V1: In-Memory (Redis)"]
        WS1["Wallet Service"]
        ZK1["ZooKeeper"]
        R1["Redis A"]
        R2["Redis B"]
        R3["Redis C"]
        WS1 --> R1
        WS1 --> R2
        WS1 --> R3
        ZK1 -.->|"Partition info"| WS1
    end

    subgraph v2["V2: Relational DB + Distributed Txn"]
        WS2["Wallet Service"]
        ZK2["ZooKeeper"]
        DB1[("DB A")]
        DB2[("DB B")]
        DB3[("DB C")]
        WS2 --> DB1
        WS2 --> DB2
        WS2 --> DB3
        ZK2 -.->|"Partition info"| WS2
    end

    subgraph v3["V3: Event Sourcing + CQRS"]
        CMD["Command Queue"]
        SM["State Machine"]
        EVT["Event Queue"]
        STATE[("State (Balance)")]
        CMD --> SM
        SM --> EVT
        EVT --> STATE
    end

    style v1 fill:#ffe6e6,stroke:#cc0000
    style v2 fill:#fff3e6,stroke:#cc6600
    style v3 fill:#e6ffe6,stroke:#006600
```

**V1 problem:** Redis is not durable; two updates across nodes are not atomic.
**V2 problem:** Transactions work but provide no audit trail; cannot replay history.
**V3 solution:** Immutable event log + deterministic state machine = full reproducibility.

## Deep Dive: Distributed Transactions

```mermaid
sequenceDiagram
    participant C as Coordinator
    participant A as Database A (Account A)
    participant B as Database C (Account C)

    Note over C,B: 2PC: Both phases in same transaction
    C->>A: Prepare (A - )
    C->>B: Prepare (C + )
    A-->>C: OK (locked)
    B-->>C: OK (locked)
    C->>A: Commit
    C->>B: Commit

    Note over C,B: TC/C: Each phase is separate transaction
    rect rgb(230, 245, 255)
        C->>A: Try: A -  (committed)
        C->>B: Try: NOP
        A-->>C: Done
        B-->>C: Done
    end
    rect rgb(230, 255, 230)
        C->>A: Confirm: NOP
        C->>B: Confirm: C +  (committed)
        A-->>C: Done
        B-->>C: Done
    end
```

| Feature | 2PC | TC/C | Saga |
|---|---|---|---|
| Lock scope | Held across phases | Released after each phase | Released after each phase |
| Coordinator failure | Databases stuck holding locks | Phase status table enables recovery | Phase status table enables recovery |
| Execution order | Any | Any (parallel possible) | Strictly linear |
| Rollback mechanism | Database-level abort | Application-level compensating txn | Application-level compensating txn |
| Performance | Slow (long locks) | Faster (short locks) | Medium (sequential) |
| Best for | Simple cases | Latency-sensitive + many services | Microservice architecture |

### TC/C: Valid Operation Orders

Only **deduct-first** is safe. If you credit first and the debit fails, someone may spend the credited money before you can reverse it.

### Out-of-Order Handling

Network delays may cause a Cancel to arrive before the Try. Solution: the Cancel sets an out-of-order flag; the subsequent Try checks this flag and returns failure.

## Deep Dive: Event Sourcing

```mermaid
graph LR
    subgraph write["Write Path"]
        CQ["Command Queue (Kafka)"]
        SM1["State Machine: Validate"]
        EQ["Event Queue"]
        SM2["State Machine: Apply"]
        ST[("State (Balances)")]
    end

    subgraph read["Read Path (CQRS)"]
        EQ2["Published Events"]
        RSM1["Read SM: Balance Query"]
        RSM2["Read SM: Audit Trail"]
        RSM3["Read SM: Investigation"]
    end

    CQ -->|"1. Read cmd"| SM1
    SM1 -->|"2. Validate"| ST
    SM1 -->|"3. Generate event"| EQ
    EQ -->|"4. Apply"| SM2
    SM2 -->|"5. Update"| ST
    EQ -->|"Publish"| EQ2
    EQ2 --> RSM1
    EQ2 --> RSM2
    EQ2 --> RSM3

    style SM1 fill:#b3e6b3,stroke:#333
    style SM2 fill:#b3e6b3,stroke:#333
```

**Key terms:**
- **Command:** Intended action ("transfer dollar 1 from A to C") -- may be invalid
- **Event:** Validated fact ("transferred dollar 1 from A to C") -- must be executed
- **State:** Current balances (key-value map)
- **State machine:** Deterministic processor; no randomness, no external I/O

**Reproducibility:** Replay the immutable event list with the deterministic state machine to reconstruct any historical state identically every time.

## Deep Dive: High-Performance File-Based Architecture

```mermaid
graph TB
    subgraph memory["Memory Layer"]
        CL["Command List (cached)"]
        EL["Event List (cached)"]
        RC["RocksDB Cache"]
    end

    subgraph disk["Disk Layer"]
        CF["Command File (mmap)"]
        EF["Event File (mmap)"]
        RDB["RocksDB (LSM-tree)"]
    end

    subgraph snapshot["Snapshot Layer"]
        HDFS["HDFS / Object Storage"]
    end

    CL <-->|"mmap"| CF
    EL <-->|"mmap"| EF
    RC <--> RDB
    RDB -->|"Periodic snapshot"| HDFS

    style memory fill:#e6f3ff,stroke:#333
    style disk fill:#fff3e6,stroke:#333
    style snapshot fill:#f0e6ff,stroke:#333
```

**Three optimizations:**
1. **mmap** for commands/events -- sequential disk writes cached in memory automatically
2. **RocksDB** for state -- LSM-tree optimized for writes; recent data cached
3. **Snapshots** to HDFS -- avoid replaying from the beginning; start from checkpoint

## Deep Dive: Raft Consensus for Reliability

```mermaid
graph TB
    subgraph raft["Raft Node Group (3 nodes)"]
        Leader["Leader"]
        F1["Follower 1"]
        F2["Follower 2"]
    end

    CMD2["Commands"] --> Leader
    Leader -->|"Replicate events"| F1
    Leader -->|"Replicate events"| F2
    Leader -->|"Apply events"| LS[("Leader State")]
    F1 -->|"Apply events"| FS1[("Follower 1 State")]
    F2 -->|"Apply events"| FS2[("Follower 2 State")]

    style Leader fill:#b3e6b3,stroke:#333
    style F1 fill:#ffe6b3,stroke:#333
    style F2 fill:#ffe6b3,stroke:#333
```

**Only event data needs replication** -- state and snapshots are regenerated from events. With 3 nodes, tolerates 1 failure. With 5 nodes, tolerates 2 failures.

**Leader crash:** Raft auto-elects new leader. Client retries the command if it was not yet converted to an event.

## Deep Dive: Final Distributed Architecture

```mermaid
graph TB
    User["User"] --> SC["Saga Coordinator"]
    SC --> PT[("Phase Status Table")]

    subgraph p1["Partition 1 (Account A)"]
        RP1["Reverse Proxy"]
        RNG1["Raft Node Group"]
        RP1 <--> RNG1
    end

    subgraph p2["Partition 2 (Account C)"]
        RP2["Reverse Proxy"]
        RNG2["Raft Node Group"]
        RP2 <--> RNG2
    end

    SC -->|"1. A:-dollar 1"| RP1
    RP1 -->|"2. Push result"| SC
    SC -->|"3. C:+dollar 1"| RP2
    RP2 -->|"4. Push result"| SC
    SC -->|"5. Done"| User

    style SC fill:#ffcc99,stroke:#333
    style RNG1 fill:#b3e6b3,stroke:#333
    style RNG2 fill:#b3e6b3,stroke:#333
```

Each partition is a full Raft group running file-based event sourcing with CQRS. The reverse proxy provides synchronous-feeling responses by receiving push notifications from the read-only state machine.

## Trade-offs

| Decision | Pro | Con |
|---|---|---|
| Event sourcing over state-based | Full auditability; reproducible history | More storage; replay overhead |
| File-based (mmap + RocksDB) over remote DB | Max I/O throughput; no network latency | Stateful nodes; need Raft for reliability |
| Raft consensus | Tolerates minority failures; auto-recovery | Write overhead for replication; odd-number nodes |
| Saga over TC/C | Simpler; microservice standard | Sequential only; higher latency |
| TC/C over Saga | Parallel execution; lower latency | Complex compensation logic |
| CQRS | Multiple read views; decoupled scaling | Eventual consistency on reads |
| Snapshots | Fast recovery; efficient point-in-time queries | Snapshot storage cost; periodic pauses |

## Takeaways

1. **Event sourcing is the de-facto standard** for financial systems requiring audit and reproducibility
2. **Immutable event log + deterministic state machine** = guaranteed identical replay
3. **File-based storage (mmap + RocksDB)** dramatically outperforms remote database access
4. **Raft consensus** ensures reliability without sacrificing event sourcing benefits
5. **CQRS** separates read/write concerns and enables multiple query-optimized views
6. **Distributed transactions** (TC/C or Saga) are needed when data spans partitions
7. **Deduct-first** is the only safe operation order in TC/C to prevent money creation bugs

## Related Concepts

- [[event-sourcing]] -- Immutable event log architecture for auditability
- [[cqrs]] -- Read/write path separation with event publishing
- [[distributed-transactions]] -- 2PC, TC/C, and Saga patterns
- [[raft-consensus]] -- Consensus algorithm for event replication
- [[high-performance-event-sourcing]] -- mmap, RocksDB, and snapshot optimizations
- [[digital-wallet-architecture]] -- Final partitioned design with Saga coordination
- [[audit-trail]] -- Reproducibility and compliance verification